In [ ]:
import spacy
import scispacy
from scispacy.linking import EntityLinker
from scispacy.abbreviation import AbbreviationDetector
import csv
from textblob import TextBlob
from textblob.np_extractors import ConllExtractor


ModuleNotFoundError: No module named 'click'

In [ ]:
nlp = spacy.load("en_core_sci_sm")
nlp.add_pipe("scispacy_linker", config={"linker_name": "umls"})
nlp.add_pipe("abbreviation_detector")

c:\Users\amara\Documents\personal-health-passport\mhp-env\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\amara\Documents\personal-health-passport\mhp-env\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:

SEMANTIC_TYPES = {}

with open("semantic_types.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        SEMANTIC_TYPES[row["code"]] = row



In [ ]:
text = """There is evidence of cancer."""

text2 = """
History:
The patient has lupus nephritis and hypertension. She was started on mycophenolate mofetil (MMF) and prednisone six months ago.

Assessment:
Proteinuria has significantly improved since treatment began. Renal function remains stable. The patient's fatigue has decreased, but the joint pain has worsened over the past two weeks. Blood pressure is well controlled. 
Serum creatinine has increased slightly compared with the previous visit. The rash has completely resolved. 
The patient denies fever, chest pain, and shortness of breath. Mild nausea developed after increasing the MMF dose but has since subsided.

Plan:
Continue MMF and prednisone. Reduce the steroid dose gradually. Repeat renal function tests in four weeks.
"""

text3 = """ The patient's cough has improved, although the dyspnoea continues to worsen. 
Peripheral oedema has almost completely resolved, while serum creatinine remains elevated. 
There is no evidence of infection, and the patient denies fever. 
Methotrexate was discontinued because liver enzymes increased significantly. 
Overall, rheumatoid arthritis appears to be responding well to treatment.
"""

text4 = """
Assessment

The patient has biopsy-proven lupus nephritis and is currently receiving mycophenolate mofetil and rituximab.

Overall, rheumatoid arthritis appears to be responding well to treatment. Proteinuria has significantly improved since the previous visit, although renal function remains stable. Serum creatinine is unchanged.

Dyspnoea continues to worsen despite corticosteroid therapy, and mild ankle oedema has developed over the past week.

There is no evidence of active infection, and the patient denies fever, chest pain, shortness of breath, abdominal pain, or haematuria.

The skin rash has completely resolved, but intermittent joint stiffness persists. Fatigue remains mildly improved following treatment.

Blood pressure is well controlled on the current medication regimen.

Plan

Continue mycophenolate mofetil. 
Reduce the prednisolone dose if renal function remains stable. 
Repeat renal function tests and urine protein-to-creatinine ratio in four weeks. 
Monitor dyspnoea closely and consider chest CT if symptoms continue to worsen.
"""

doc = nlp(text)
doc2 = nlp(text2)
doc3 = nlp(text3)
doc4 = nlp(text4)

In [ ]:
def get_semantic_class(code):
    info = SEMANTIC_TYPES[code]
    return info["category"]

In [ ]:
def get_entity_info(doc):
    entities = {}
    linker = nlp.get_pipe("scispacy_linker")

    abbreviation_map = {
        str(abbr): str(abbr._.long_form)
        for abbr in doc._.abbreviations
    }

    print("Abbreviation Map:", abbreviation_map)
    
    for entity in doc.ents:
        entity_text = entity.text

        normalised_text = abbreviation_map.get(entity_text, entity_text)

        normalised_doc = nlp(normalised_text)

        if normalised_doc.ents:
            normalised_entity = normalised_doc.ents[0]

            if normalised_entity._.kb_ents:
                cui, score = normalised_entity._.kb_ents[0]
                concept = linker.kb.cui_to_entity[cui]

                entities[entity_text] = {
                    "entity": entity,
                    "tokens": [x for x in entity_text.split()],
                    "normalised": normalised_text,
                    "canonical": concept.canonical_name,
                    "semantic types": [get_semantic_class(x) for x in concept.types],
                    "score": score
                    
                }

    return entities

get_entity_info(doc2)

Abbreviation Map: {'MMF': 'mycophenolate mofetil'}


{'patient': {'entity': patient,
  'tokens': ['patient'],
  'normalised': 'patient',
  'canonical': 'Patients',
  'semantic types': ['Living Beings'],
  'score': 0.9829968214035034},
 'lupus nephritis': {'entity': lupus nephritis,
  'tokens': ['lupus', 'nephritis'],
  'normalised': 'lupus nephritis',
  'canonical': 'Lupus Nephritis',
  'semantic types': ['Disorders'],
  'score': 0.9638109803199768},
 'hypertension': {'entity': hypertension,
  'tokens': ['hypertension'],
  'normalised': 'hypertension',
  'canonical': 'Hypertensive disease',
  'semantic types': ['Disorders'],
  'score': 0.9838637709617615},
 'months': {'entity': months,
  'tokens': ['months'],
  'normalised': 'months',
  'canonical': 'month',
  'semantic types': ['Concepts & Ideas'],
  'score': 0.9597114324569702},
 'Assessment': {'entity': Assessment,
  'tokens': ['Assessment'],
  'normalised': 'Assessment',
  'canonical': 'Physical Examination',
  'semantic types': ['Procedures'],
  'score': 0.9750187993049622},
 'Prote

In [ ]:
def attach_entity_relationship(doc,ents):

    for ent in ents: 
        index = ents[ent]["entity"].end - 1
        ctoken = doc[index]

        for child in ctoken.children:    
            phrase = ctoken.text 
            if not child.is_space and child.dep_ in ("amod", "compound" ):
                phrase = child.text + " " + phrase
            print(phrase)
                
attach_entity_relationship(doc2, get_entity_info(doc2))
            

Abbreviation Map: {'MMF': 'mycophenolate mofetil'}


c:\Users\amara\Documents\personal-health-passport\mhp-env\Lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)


patient
lupus nephritis
nephritis
nephritis
months
Assessment
Assessment
Assessment
Assessment
Renal function
fatigue
decreased
decreased
decreased
decreased
decreased
decreased
pain
joint pain
worsened
worsened
worsened
weeks
weeks
Blood pressure
creatinine
Serum creatinine
increased
increased
increased
increased
increased
visit
visit
previous visit
rash
denies
denies
denies
denies
denies
denies
denies
denies
chest pain
pain
shortness
Mild nausea
increasing
increasing
dose
MMF dose
dose
steroid dose
Repeat tests
function tests
tests
tests


In [ ]:
def check_negation(token):
    negation_words = {
        "deny",
        "no",
        "without",
        "negative",
        "absence"
    }

    if token.dep_ == "neg" or token.lemma_.lower() in negation_words:
        for child in token.children:
            if child.dep_ == "dobj":
                return {child.text,token.lemma_.lower()}
            
    return False

In [ ]:
def nmod_context_extraction(doc):

    ids = []

    for token in doc:
        negated = False

        if token.dep_ == "nsubj" or token.dep_ == "attr" or token.dep_ == "conj":
            for child in token.children:
                if child.text == "no":
                    negated = True
                
                if child.dep_ == "nmod":                  
                    ids.append({
                        "nsubj" : token.i,
                        "nmod" : child.i,
                        "negated" :negated
                        })
     
    return ids

nmod_context_extraction(doc)

[{'nsubj': 2, 'nmod': 4, 'negated': False}]

In [ ]:
def get_conj_entities(token):
    entities = []

    for child in token.children:
        if child.dep_ == "conj":
            entities.append(child)
            entities.extend(get_conj_entities(child))

    return entities

In [ ]:
def print_tokens(doc):
    for token in doc:
        print(
            f"{token.i:<15}"
            f"{token.text:<15}"
            f"POS={token.pos_:<6}"
            f"DEP={token.dep_:<12}"
            f"HEAD={token.head.text}"
        )

print_tokens(doc2)


In [ ]:
def extract_clinical_relationships(doc):
    improvement_words = {
        "improve",
        "resolve",
        "decrease",
        "reduce",
        "subside",
        "recover",
        "remit"
    }

    decline_words = {
        "worse",
        "worsen",
        "deteriorate",
        "decline",
        "progress",
        "exacerbate",
        "increase",
        "rise",
        "relapse",
        "develop",
        "discontinue",
        "elevated"
    }

    neutral_words = {
        "respond"
    }

    positive_mods = {
        "well",
        "significantly",
        "markedly",
        "completely",
        "fully",
        "adequately",
        "favourably",
        "favorably",
    }

    negative_mods = {
        "poorly",
        "badly",
        "minimally",
        "slightly",
        "partially",
        "inadequately",
        "suboptimally",
    }
    stable_words = {
        "stable",
        "remain",
        "unchanged",
        "control",
        "continue",
        "persist"
    }

    negation_words = {
        "deny",
        "no",
        "without",
        "negative",
        "absence"
    }

    relationships = []

    nmod_phrase_ids = nmod_context_extraction(doc)

    subjects = []


    for token in doc:
        
        lemma = token.lemma_.lower()

        # Find clinical status verbs
        if lemma in improvement_words | decline_words | stable_words | negation_words | neutral_words:

            status = None

            if lemma in improvement_words:
                status = "improved"
            elif lemma in decline_words:
                status = "declined"

            elif lemma in stable_words:
                status = "stable"
                for child in token.children:
                    if child.lemma_.lower() in improvement_words:
                        status = "improved"
                    elif child.lemma_.lower() in decline_words:
                        status = "declined"

            elif lemma in negation_words:
                status = "negated"

            # Find the entity connected to the verb
            if token.text == "no": 
                for child in token.children:
                    print(child.text, child.dep_, child.pos_)

            for child in token.children:

                if child.dep_ in ("dobj","nsubj","nsubjpass") :

                    subjects.append({
                        "token": child,
                        "trigger": token.text,
                        "status": status
                    })


                    for conj in token.conjuncts:
                        if conj.pos_ in ("NOUN", "PROPN"):
                            print(f"Parent Entity: {token.text}, Conjunct Entity: {conj.text}")
                            subjects.append({
                                "token": conj,
                                "trigger": token.text,
                                "status": status
                            })
                            #if status == "negated" and child.dep_ == "nsubj": print(f"Negated Entity: {child.text}, Trigger: {token.text}")                
                elif token.dep_ == "xcomp" or token.dep_ == "ccomp":
                    print(token.text)        
                    #print(child.text)
                    head = token.head
                    trigger = ""
                    

                    for child in token.children:
                        if child.dep_ == "advmod" :
                            trigger = token.text + " " + child.text
                            if child.lemma_ in positive_mods or child.lemma_ in improvement_words:
                                status = "improved"
                            elif child.lemma_ in negative_mods  or child.lemma_ in decline_words:
                                status = "declined"

                            break
                    
                    for child in head.children:
                        if child.dep_ in ("nsubj", "nsubjpass"):
                            #print(child.text)
                            subjects.append({
                                "token": child,
                                "trigger": trigger if trigger != "" else token.text,
                                "status": status
                            })
                            break

        else:          
            for phrase in nmod_phrase_ids:               
                if token.i == phrase["nsubj"]:
                    relationships.append({
                            "entity": f"{doc[phrase["nsubj"]]} of {doc[phrase["nmod"]]}" ,
                            "trigger": "remains" if phrase["negated"] == False else "no" ,
                            "status": "stable" if phrase["negated"] == False else "negated"
                            })

    for sub in subjects:
        phrase = sub["token"].text
        modifiers = ""
        for grandchild in sub["token"].children:
            if not grandchild.is_space and grandchild.dep_ in ("amod", "compound" ):
                modifiers += grandchild.text + " "
                #if grandchild.text == "Overall" : print(phrase)
            elif not grandchild.is_space and grandchild.dep_ in ("nmod"):
                phrase = f"{phrase} of {grandchild.text}"
                #if grandchild.text == "Overall" : print(phrase)
                break

        if modifiers != "" : phrase = modifiers + " " + phrase 
        
        
        relationships.append({
            "entity": phrase,
            "trigger": sub["trigger"],
            "status": sub["status"]
        })

    return relationships
        
for relationship in extract_clinical_relationships(doc2):
    print(relationship)


Parent Entity: denies, Conjunct Entity: pain
Parent Entity: denies, Conjunct Entity: shortness
Parent Entity: denies, Conjunct Entity: pain
Parent Entity: denies, Conjunct Entity: shortness
{'entity': 'shortness of breath', 'trigger': 'remains', 'status': 'stable'}
{'entity': 'Proteinuria', 'trigger': 'improved', 'status': 'improved'}
{'entity': 'Renal  function', 'trigger': 'remains', 'status': 'stable'}
{'entity': 'fatigue', 'trigger': 'decreased', 'status': 'improved'}
{'entity': 'joint  pain', 'trigger': 'worsened', 'status': 'declined'}
{'entity': 'Blood  pressure', 'trigger': 'controlled', 'status': 'stable'}
{'entity': 'Serum  creatinine', 'trigger': 'increased', 'status': 'declined'}
{'entity': 'rash', 'trigger': 'resolved', 'status': 'improved'}
{'entity': 'patient', 'trigger': 'denies', 'status': 'negated'}
{'entity': 'chest  pain', 'trigger': 'denies', 'status': 'negated'}
{'entity': 'shortness of breath', 'trigger': 'denies', 'status': 'negated'}
{'entity': 'fever', 'trigge

The patient's cough has improved, although the dyspnoea continues to worsen. 
Peripheral oedema has almost completely resolved, while serum creatinine remains elevated. 
There is no evidence of infection, and the patient denies fever. 
Methotrexate was discontinued because liver enzymes increased significantly. 
Overall, rheumatoid arthritis appears to be responding well to treatment.